In [ ]:
import xgboost as xgb

In [ ]:
from scipy.stats import uniform
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import make_scorer
from sklearn.grid_search import RandomizedSearchCV, GridSearchCV
from sklearn.cross_validation import train_test_split, StratifiedKFold
from xgboost import XGBRegressor
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline  
plt.rcParams['figure.figsize'] = [15, 5]

In [ ]:
def timer(start_time=None):
    if not start_time:
        start_time = datetime.now()
        return start_time
    elif start_time:
        thour, temp_sec = divmod((datetime.now() - start_time).total_seconds(), 3600)
        tmin, tsec = divmod(temp_sec, 60)
        print('\n Time taken: %i hours %i minutes and %s seconds.' % (thour, tmin, round(tsec, 2)))


In [ ]:
data = pd.read_csv('results.csv')

In [ ]:
data['outcome'] = data['test_out'].apply(str) + data['control_out'].apply(str)

In [ ]:
data.head()

In [ ]:
data.columns

In [ ]:
features = ['f0','f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11']
target   = 'outcome'

In [ ]:
br = pd.DataFrame(data['outcome'].value_counts()).reset_index()

In [ ]:
br

In [ ]:
# as we can see the 00 (lost causes) segment is hugely overrepresented 

In [ ]:
# sampling strategy.
# balance by undersampling
# binary between sleeping dogs and persuadables 
# convert to regression 01 lowest to 11 highest
# three class: persuadbles, (00,11), sleeping dogs: balance 

In [ ]:
# undersampling
br = pd.DataFrame(data['outcome'].value_counts()).reset_index()

sample = pd.concat([data[data.outcome == '00'][features + [target]].sample(n=min(data['outcome'].value_counts()))
                         ,data[data.outcome == '01'][features + [target]].sample(n=min(data['outcome'].value_counts()))
                         ,data[data.outcome == '10'][features + [target]].sample(n=min(data['outcome'].value_counts()))
                         ,data[data.outcome == '11'][features + [target]].sample(n=min(data['outcome'].value_counts()))])

In [ ]:
# binary 
br = data[(data.outcome == '01') | (data.outcome == '10')]['outcome'].value_counts()

sample = pd.concat([data[data.outcome == '01'][features + [target]].sample(n=min(data['outcome'].value_counts()))
                   ,data[data.outcome == '10'][features + [target]].sample(n=min(data['outcome'].value_counts()))])



In [ ]:
# seed = 7
# test_size = 0.3
# X_trian, X_test, y_train, y_test = train_test_split(sample[features], sample[target], test_size=test_size, random_state=seed)

In [ ]:
params = {
        'min_child_weight': [1, 5, 10],
        'gamma': [0.5, 1, 1.5, 2, 5],
        'subsample': [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0],
        'max_depth': [3, 4, 5]
        }

In [ ]:
xgbc = xgb.XGBClassifier(learning_rate=0.02, n_estimators=600, objective='multi:softmax', silent=True, nthread=1, num_class=4)

In [ ]:
folds = 4
param_comb = 5

#skf = StratifiedKFold(y=folds, shuffle = True, random_state = 1001)

random_search = RandomizedSearchCV(xgbc, param_distributions=params, n_iter=param_comb, scoring='accuracy', n_jobs=4, cv=5, verbose=3, random_state=1001 )

# Here we go
#start_time = timer(None) # timing starts from this point for "start_time" variable
random_search.fit(sample[features], sample[target])
#timer(start_time) # timing ends here for "start_time" variable

In [ ]:
test = pd.read_csv('data_000000000007.csv')

In [ ]:
test.head()

In [ ]:
y_test = random_search.predict_proba(test[features])

In [ ]:
y_test

In [ ]:
y_test_pred = random_search.predict(test[features])

In [ ]:
y_test_pred

In [ ]:
outcomes = {} #'00', '01', '10', '11']:
    

In [ ]:
i = 0 
for pred in y_test_pred:
    if pred not in outcomes:
        outcomes[pred] = np.argmax(y_test[i])
    else:
        pass
    i += 1

In [ ]:
outcomes

In [ ]:
from pylift.eval import UpliftEval


In [ ]:
upev = UpliftEval(test['treatment'], test['conversion'], y_test[:,1])

In [ ]:
upev.q1_qini

In [ ]:
upev.plot(plot_type='qini',show_practical_max=True, show_no_dogs=True)